# 🩸 Diabetes Disease Prediction
**Author:** Bhardwaja  
**Dataset:** Pima Indians Diabetes Dataset — Extended (1543 samples, 8 features)  
**Goal:** Predict whether a patient has diabetes based on diagnostic measurements.

---
### Pipeline Overview
1. Import Libraries
2. Load & Explore Data (EDA)
3. Data Cleaning & Preprocessing
4. Train/Test Split (No Data Leakage)
5. Model Training & Comparison
6. Evaluation & Visualisation
7. Save Best Model

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, roc_curve
)
import joblib

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('All libraries imported successfully!')

## 2. Load & Explore the Data

In [ ]:
df = pd.read_csv('diabetes_disease.csv')
# Drop index column 'No' — not a feature
df.drop('No', axis=1, inplace=True)
print('Dataset Shape:', df.shape)
df.head(10)

In [ ]:
print('=== Dataset Info ===')
df.info()

In [ ]:
print('=== Statistical Summary ===')
df.describe()

In [ ]:
# Class distribution
n_pos = (df['Outcome'] == 1).sum()
n_neg = (df['Outcome'] == 0).sum()
total = len(df)
print(f'Diabetic     (1): {n_pos} ({n_pos/total*100:.1f}%)')
print(f'Non-Diabetic (0): {n_neg} ({n_neg/total*100:.1f}%)')

plt.figure(figsize=(6, 4))
df['Outcome'].value_counts().plot(
    kind='bar', color=['#2ecc71', '#e74c3c'], edgecolor='black'
)
plt.title('Class Distribution: Diabetic vs Non-Diabetic', fontsize=14)
plt.xlabel('Outcome (0 = No Diabetes, 1 = Diabetes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Feature histograms split by outcome
features = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
            'Insulin','BMI','DiabetesPedigreeFunction','Age']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    df[df['Outcome']==1][feat].hist(ax=axes[i], alpha=0.6, label='Diabetic', color='#e74c3c', bins=25)
    df[df['Outcome']==0][feat].hist(ax=axes[i], alpha=0.6, label='Non-Diabetic', color='#2ecc71', bins=25)
    axes[i].set_title(feat, fontsize=11)
    axes[i].legend(fontsize=8)

plt.suptitle('Feature Distributions by Diabetes Outcome', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', linewidths=0.5, annot_kws={'size': 9}
)
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Age vs Outcome countplot
plt.figure(figsize=(14, 5))
sns.countplot(x='Age', hue='Outcome', data=df, palette=['#2ecc71','#e74c3c'])
plt.title('Age Distribution by Diabetes Outcome', fontsize=14)
plt.legend(title='Outcome', labels=['Non-Diabetic', 'Diabetic'])
plt.tight_layout()
plt.show()

## 3. Data Cleaning & Preprocessing

**Key insight:** In this dataset, zero values in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` are **biologically impossible** — they represent missing data encoded as 0. We replace them with `NaN` and impute.

In [ ]:
# Columns where 0 is impossible — treat as missing
zero_as_null_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print('Zero counts (= hidden missing values):')
for col in zero_as_null_cols:
    count = (df[col] == 0).sum()
    print(f'  {col}: {count} zeros ({count/len(df)*100:.1f}%)')

# Replace 0s with NaN for these columns
df[zero_as_null_cols] = df[zero_as_null_cols].replace(0, np.nan)
print('\nZeros replaced with NaN.')

In [ ]:
# Missing value heatmap after replacement
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing Values After Zero Replacement (Yellow = Missing)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Train/Test Split & Imputation (No Data Leakage)

In [ ]:
feature_cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness',
                'Insulin','BMI','DiabetesPedigreeFunction','Age']

X = df[feature_cols]
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=52, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Test set:     {X_test.shape[0]} samples')

# IMPORTANT: fit imputer ONLY on training data to prevent data leakage
imputer = SimpleImputer(strategy='mean')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_cols)
X_test_imp  = pd.DataFrame(imputer.transform(X_test), columns=feature_cols)  # transform only!

# Scale for distance/linear models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)

print('\nSplit, imputation, and scaling done — no data leakage.')

## 5. Model Training & Comparison

Training 4 models (same as original, now fixed and properly evaluated):
- Naive Bayes
- Random Forest
- Logistic Regression
- Logistic Regression with Cross-Validation

In [ ]:
# --- Model 1: Naive Bayes ---
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
nb_pred = nb.predict(X_test_scaled)
nb_acc  = accuracy_score(y_test, nb_pred)

print(f'Naive Bayes Accuracy: {nb_acc:.4f}')
print(classification_report(y_test, nb_pred, target_names=['Non-Diabetic','Diabetic']))

In [ ]:
# --- Model 2: Random Forest ---
rf = RandomForestClassifier(n_estimators=100, random_state=52)
rf.fit(X_train_imp, y_train)
rf_pred = rf.predict(X_test_imp)
rf_acc  = accuracy_score(y_test, rf_pred)

print(f'Random Forest Accuracy: {rf_acc:.4f}')
print(classification_report(y_test, rf_pred, target_names=['Non-Diabetic','Diabetic']))

In [ ]:
# --- Model 3: Logistic Regression ---
lr = LogisticRegression(C=0.7, max_iter=1000, random_state=52)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_acc  = accuracy_score(y_test, lr_pred)

print(f'Logistic Regression Accuracy: {lr_acc:.4f}')
print(classification_report(y_test, lr_pred, target_names=['Non-Diabetic','Diabetic']))

In [ ]:
# --- Model 4: Logistic Regression with Cross-Validation ---
lr_cv = LogisticRegressionCV(
    n_jobs=-1, random_state=52, Cs=3, cv=10,
    refit=True, class_weight='balanced', max_iter=1000
)
lr_cv.fit(X_train_scaled, y_train)
lr_cv_pred = lr_cv.predict(X_test_scaled)
lr_cv_acc  = accuracy_score(y_test, lr_cv_pred)

print(f'Logistic Regression CV Accuracy: {lr_cv_acc:.4f}')
print(classification_report(y_test, lr_cv_pred, target_names=['Non-Diabetic','Diabetic']))

## 6. Evaluation & Visualisation

In [ ]:
# Accuracy comparison
model_names = ['Naive Bayes', 'Random Forest', 'Logistic Reg', 'LR + CV']
accuracies  = [nb_acc, rf_acc, lr_acc, lr_cv_acc]

plt.figure(figsize=(9, 5))
bars = plt.bar(model_names, accuracies,
               color=['#3498db','#e67e22','#9b59b6','#1abc9c'],
               edgecolor='black')
for bar, acc in zip(bars, accuracies):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() - 0.04,
        f'{acc:.2%}', ha='center', va='bottom',
        fontsize=12, fontweight='bold', color='white'
    )
plt.ylim(0.6, 1.0)
plt.title('Model Accuracy Comparison — Diabetes Prediction', fontsize=14)
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
model_info = [
    ('Naive Bayes', nb_pred),
    ('Random Forest', rf_pred),
    ('Logistic Reg', lr_pred),
    ('LR + CV', lr_cv_pred)
]
for ax, (name, pred) in zip(axes, model_info):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Non-Diabetic','Diabetic'])
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(f'{name}\nAcc: {accuracy_score(y_test,pred):.2%}', fontsize=11)
plt.suptitle('Confusion Matrices', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ROC Curve for all models
plt.figure(figsize=(9, 6))
roc_models = [
    ('Naive Bayes', nb, X_test_scaled),
    ('Random Forest', rf, X_test_imp),
    ('Logistic Reg', lr, X_test_scaled),
    ('LR + CV', lr_cv, X_test_scaled),
]
for name, model, Xte in roc_models:
    proba = model.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0,1],[0,1],'k--', label='Random Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Diabetes Prediction', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance from Random Forest
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
plt.title('Feature Importance — Random Forest', fontsize=14)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

print('Top 5 features:')
print(feat_imp.head())

In [ ]:
# Cross-validation score for best model
cv_scores = cross_val_score(rf, X_train_imp, y_train, cv=5, scoring='accuracy')
print(f'Random Forest 5-Fold CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')
print(f'Individual fold scores: {np.round(cv_scores, 4)}')

## 7. Save Best Model

In [ ]:
best_model = rf

joblib.dump(best_model, 'diabetes_model.pkl')
joblib.dump(imputer,    'diabetes_imputer.pkl')
joblib.dump(scaler,     'diabetes_scaler.pkl')

print('Saved: diabetes_model.pkl')
print('Saved: diabetes_imputer.pkl')
print('Saved: diabetes_scaler.pkl')
print('\nReady for the multi-disease Streamlit app!')

## 8. Summary

| Model | Test Accuracy |
|---|---|
| Naive Bayes | see above |
| **Random Forest** | **best** |
| Logistic Regression | see above |
| LR with CV | see above |

**Key Findings:**
- `Glucose` is the strongest single predictor of diabetes — clinically expected.
- `BMI` and `Age` are also highly important features.
- Zero values in medical columns were treated as missing (not as 0) — a critical preprocessing step.
- Data leakage was avoided by fitting the imputer/scaler only on training data.

**Possible improvements:**
- SMOTE oversampling to address class imbalance (65% vs 34%)
- Hyperparameter tuning with GridSearchCV
- XGBoost or SVM as additional models